In [ ]:
from google.cloud import bigquery, storage

# we will use the storage client only for demonstration purposes
storage_client = storage.Client(
    project=PROJECT_ID, client_info=ClientInfo(user_agent=USER_AGENT)
)

# we will use the bigquery client to prepare an empty table, backed by apache iceberg parquet format.
bigquery_client = bigquery.Client(
    project=PROJECT_ID, location=LOCATION, client_info=ClientInfo(user_agent=USER_AGENT)
)

## Let's take a look around
We've created a spark session, with a couple of interesting connections. We're running a spark session with the spark-bigquery-connector, which gives us access to all BigQuery tables and Views, and also configured Iceberg catalogs (the one managed by BigLake, and the REST for managing external tables)

Let's take a quick look at the **catalogs** - each containig **namespses** (also known as databases) and each namespace has **tables**. And compare the access patterns,

Can you see the difference?

There are a few differences between the 2 tables, and the way our spark reads the data. Although reading the data would produce the same logical dataset, it matters how get that data.

Note that the spark-bigquery-connector version (using `spark_catalog`) says in the `Table Properties` section, that the storage layer is set to `bigquery`. This implies that in order to access the data with this table, we will go though BigQuery to get the data.

Note in the second description, the `Table Properties` doesn't say that, rather says that the format is `iceberg/parquet`, and we do get the exact GCS location of the data as well. This implies that to access the data, we just let the spark-iceberg library access the data directly from GCS.

The way we choose to read our data has a big impact on the pricing of access to the data, since accessing it through BQ will impact BQ slots usage.

With that in mind, let's read all the datasets required, each with a different method.

In [ ]:
# To read the bus_stations data, we will use reading directly from bigquery, which uses the underlying bigquery-spark connector and uses bigquery slots to read
bus_stations_df = spark.read.format("bigquery").load(
    f"{PROJECT_ID}.{BQ_DATASET}.bus_stations"
)
bus_stations_df.printSchema()
bus_stations_df.show(5)

In [ ]:
# join back to the full bus line data, to get the full context of each bus line
bus_lines_with_min_max = bus_line_overall_times_df.join(bus_lines_df, on="bus_line_id")
bus_lines_with_min_max.show(10)

In [ ]:
# join with the ridership data, by each station_id and the timestamp
bus_rides_with_ridership = bus_rides.alias("rides").join(
    ridership_df.alias("ridership"),
    (f.col("rides.bus_stop_id") == f.col("ridership.station_id"))
    & (f.col("rides.timestamp_at_stop") == f.col("ridership.transit_timestamp")),
    "inner",
)

# drop redundant columns and rename `ridership` to `passengers_in_stop` for clarity
bus_rides_with_ridership = (
    bus_rides_with_ridership.drop("transit_timestamp")
    .drop("station_id")
    .withColumnRenamed("ridership", "passengers_in_stop")
)
# cache this, as this will be used again shortly
bus_rides_with_ridership.cache()
bus_rides_with_ridership.show(10)

## Store results to bigquery

In [ ]:
# save the bus_rides_with_riders_and_totals dataframe to bigquery, with iceberg format
bus_rides_with_riders_and_totals.write.format("bigquery").mode("overwrite").option(
    "table", f"{PROJECT_ID}.{BQ_DATASET}.{TABLE_NAME}"
).save()